# 1. SIMD programming (I) --- high level SIMD programming with auto vectorization and OpenMP SIMD directives
* In this notebook you learn how to use SIMD instructions of CPUs without explicitly manipulating vectors of values (i.e., let the compiler vectorize ordinary loops only using scalar values)

# 2. Basics
* <font color="blue">SIMD</font> stands for _Single Instruction Multiple Data_
* CPU offers <font color="blue">SIMD registers</font> that can hold a number of operands (say 16 float numbers) as well as <font color="blue">SIMD instructions</font> that operate on all values on SIMD registers
* In the context of SIMD programming, we use "SIMD" and <font color="blue">"vector"</font> almost interchangeably.  We say "vector registers" or "vector operands" in place of "SIMD registers" or "SIMD operands"
* We say "<font color="blue">vectorize</font> this program" to mean "utilize vector/SIMD instructions".  We casually say "<font color="blue">simdize</font> this program" as well.
* We call an ordinary single value (e.g., a float) a <font color="blue">"scalar"</font>.  Scalar is an antonym of vector in SIMD programming context.
* There are various ways to vectorize programs, roughly in the order of high-level approaches to low-level ones
  * use libraries that are already using SIMD (we don't discuss this further)
  * write ordinary loops and hope the compiler does the job
  * use simd directives of OpenMP or other language constructs that are designed to be compiled to SIMD instructions but never make SIMD operands explicit in the program; all expressions of your program are still of ordinary scalar types
  * use explicit vector types and variables/expressions of vector types, effectively making "SIMD registers" visible entities in programs
  * plus use vector intrinsics, functions that almost directly correspond to SIMD instructions
  * use assembly

# 3. High-level SIMD programming and vectorization report
* in so-called "high level" SIMD programming, you do not explicitly deal with SIMD values nor instructions
* you typically write an ordinary loop and hope the compiler is able to vectorize it
* in this approach, the basic issue is how to know if the compiler is successfully able to vectorize your code
* there are options to ask the compiler to report about successful/missed vectorizations


# 4. Compilers
* Modern compilers including GCC, LLVM, NVIDIA HPC SDK support SIMD.
* We continue to use [NVIDIA HPC SDK ver. 22.9](https://docs.nvidia.com/hpc-sdk/index.html) (`nvc` and `nvc++`) and [LLVM ver. 15.0](https://llvm.org/) (`clang` and `clang++`)
* We also try GCC to see if which compilers are able to vectorize which code

## 4-1. Set up NVIDIA HPC SDK
Execute this before you use NVIDIA HPC SDK

In [1]:
export PATH=/opt/nvidia/hpc_sdk/Linux_x86_64/22.9/compilers/bin:$PATH
#export PATH=/opt/nvidia/hpc_sdk/Linux_x86_64/22.7/compilers/bin:$PATH

Check if it works (check if full paths of nvc/nvc++ are shown)

In [2]:
which nvc
which nvc++

/opt/nvidia/hpc_sdk/Linux_x86_64/22.9/compilers/bin/nvc
/opt/nvidia/hpc_sdk/Linux_x86_64/22.9/compilers/bin/nvc++


## 4-2. Set up LLVM
Execute this before you use LLVM

In [72]:
export PATH=/home/share/llvm/bin:$PATH

Check if it works (check if full paths of nvc/nvc++ are shown)

In [71]:
which clang
which clang++

/home/share/llvm/bin/clang
/home/share/llvm/bin/clang++


## 4-3. GCC
* It's installed at a usual location /usr/bin/gcc

Check if it works (check if full paths of nvc/nvc++ are shown)

In [5]:
which gcc
which g++

/usr/bin/gcc
/usr/bin/g++


# 5. A first encounter to a vectorized code
* let's take a look at a code easiest to vectorize
* serious compilers generally support options to let the programmer know how they vectorized (or didn't vectorize) the code
  * clang(++): `-Rpass=loop-vectorize` and `-Rpass-missed=loop-vectorize`
  * nvc(++): `-Minfo=vect` and `-Mneginfo=vect`
  * gcc (g++): `-fopt-info-vec-optimized` and `-fopt-info-vec-missed`

* you'll see the following code is successfully vectorized with sufficiently high optimization flags (e.g., `-O`, `-O3`, or `-O4`)

In [7]:
%%writefile axpb.c
void axpb(float a, float * x, float b, long n) {
  for (long i = 0; i < n; i++) {
    x[i] = a * x[i] + b;
  }
}

Overwriting axpb.c


In [8]:
# will produce axpb.s
clang -S -Wall -O3 -mavx512f -mfma -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpb.c
#nvc -S -Wall -O3 -mavx512f -mfma -Minfo=vect -Mneginfo=vect axpb.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopt-info-vec-optimized -fopt-info-vec-missed axpb.c

axpb.c:2:3: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
  for (long i = 0; i < n; i++) {
  ^


* roughly, vectorization is about converting a loop like
```
for (i = 0; i < n; i++) {
  statement(i);
}
```
into
```
for (i = 0; i < n; i += L) {
  statement(i:i+L);
}
```
where L is the SIMD width (we assume L divides n) and statement(i:i+L) is an informal notation that executes statement(i) ... statement(i+L-1).

The above loop will be executed like
```
for (i = 0; i < n; i += L) {
  x[i:i+L] = a * x[i:i+L] + b;
}
```
where `x[i:i+L]` represents a vector of L values `x[i], x[i+1], ..., x[i+L-1]`.

# 6. Instructions and compiler flags
* SIMD instructions are used when an optimization level is beyond a certain threshold or a specific flag is given
  * clang : `-fvectorize`
  * nvc : `-Mvect`
  * gcc : `-ftree-vectorize`
* which SIMD instructions are used depends on compiler flags and the host CPU
  * `-mavx` uses AVX (256 bits)
  * `-mavx2` uses AVX2 (256 bits)
  * `-mavx512f` uses AVX512F (512 bits)
  * `-mfma` uses fused multiply-add
  * `-march=native` should use instructions supported on the compilation host, but do not count on this (see below)
* by giving a suitable flag, it is possible to generate instructions regardless of the instructions supported on the compilation host. e.g., `-mavx512f` will generate AVX512F instructions even on hosts not supporting them
* you can check which ISAs are supported by looking at `/proc/cpuinfo`
* taulec and tauleg000 support AVX512F (and its ancestors AVX2, AVX, and SSE)
* giving `-march=native` on taulec should use AVX512F, but it doesn't, for reasons I don't know. let's stick with `-mavx512f -mfma` (you can change it to other flags when necessary)

In [9]:
grep flags /proc/cpuinfo | uniq

flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc arch_perfmon rep_good nopl xtopology tsc_reliable nonstop_tsc cpuid pni pclmulqdq vmx ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt tsc_deadline_timer aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch cpuid_fault invpcid_single ssbd ibrs ibpb stibp ibrs_enhanced tpr_shadow vnmi ept vpid ept_ad fsgsbase tsc_adjust bmi1 avx2 smep bmi2 erms invpcid avx512f avx512dq rdseed adx smap avx512ifma clflushopt clwb avx512cd sha_ni avx512bw avx512vl xsaveopt xsavec xgetbv1 xsaves wbnoinvd arat avx512vbmi umip pku ospke avx512_vbmi2 gfni vaes vpclmulqdq avx512_vnni avx512_bitalg avx512_vpopcntdq rdpid md_clear flush_l1d arch_capabilities


* if you run an executable that uses unsupported instructions, you will get an illegal instruction
* the following should run OK on our environment (taulec, tauleg000, or some other hosts that come out later)

In [10]:
%%writefile test_isa_avx512f.c
#include <stdio.h>
#include <x86intrin.h>
int main(int argc, char ** argv) {
  int i = 1;
  float a = (argc > i ? atof(argv[i]) : 1.23); i++;
  float b = (argc > i ? atof(argv[i]) : 1.23); i++;
  __m512 v = _mm512_set1_ps(b);
  __m512 c = a * v;
  printf("OK: c[0] = %f\n", c[0]);
  return 0;
}

Writing test_isa_avx512f.c


In [11]:
clang -O3 -Wall -mavx512f -mfma -o test_isa_avx512f test_isa_avx512f.c
#nvc -O3 -Wall -mavx512f -mfma -o test_isa_avx512f test_isa_avx512f.c
#gcc -O3 -Wall -mavx512f -mfma -o test_isa_avx512f test_isa_avx512f.c

In [12]:
./test_isa_avx512f

OK: c[0] = 1.512900


# 7. `-S` option is your friend
* let's compare the generated machine code (assembly) of vectorized and unvectorized versions
* here are things you want to know
  1. `-S` option (of clang, nvc and gcc) generates assembly code (_xxx_.c becomes _xxx_.s)
  1. `asm volatile ("...")` is an inline assembly, which simply puts ... into the generated assembly code
  1. its true purpose is to write assembly instruction in C code, but here we use it to put a landmark in the beginning and the end of a loop we are interested in.  note that we put `asm volatile ("# ...")`, which is a comment of assembly having no effect. without it it quickly becomes difficult to locate the assembly code corresponding to the loop

## 7-1. inhibiting vectorization
* compilers generally support options to disable vectorization
  * clang : `-fno-vectorize`
  * nvc : ??? (`-Mnovect` should be it, but it does not seem to have the desired effect)
  * gcc : `-fno-tree-vectorize`

In [13]:
%%writefile axpb_scalar.c
void axpb_scalar(float a, float * x, float b, long n) {
  asm volatile("# ========== loop begins ==========");
  for (long i = 0; i < n; i++) {
    x[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing axpb_scalar.c


* with `-S` option you can see the generated instructions (machine code or assembly code to be more precise)

In [14]:
# will produce axpb_scalar.s
clang -S -Wall -O3 -mavx512f -mfma -fno-vectorize -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpb_scalar.c
#nvc -S -Wall -O3 -mavx512f -mfma -Mnovect -Minfo=vect -Mneginfo=vect axpb_scalar.c
#gcc -S -Wall -O3 -mavx512f -mfma -fno-tree-vectorize -fopt-info-vec-optimized -fopt-info-vec-missed axpb_scalar.c

axpb_scalar.c:3:3: remark: loop not vectorized [-Rpass-missed=loop-vectorize]
  for (long i = 0; i < n; i++) {
  ^


In [15]:
cat -n axpb_scalar.s

     1		.text
     2		.file	"axpb_scalar.c"
     3		.globl	axpb_scalar                     # -- Begin function axpb_scalar
     4		.p2align	4, 0x90
     5		.type	axpb_scalar,@function
     6	axpb_scalar:                            # @axpb_scalar
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		testq	%rsi, %rsi
    13		jle	.LBB0_6
    14	# %bb.1:
    15		leaq	-1(%rsi), %rcx
    16		movl	%esi, %eax
    17		andl	$3, %eax
    18		cmpq	$3, %rcx
    19		jae	.LBB0_7
    20	# %bb.2:
    21		xorl	%ecx, %ecx
    22		jmp	.LBB0_3
    23	.LBB0_7:
    24		andq	$-4, %rsi
    25		xorl	%ecx, %ecx
    26		.p2align	4, 0x90
    27	.LBB0_8:                                # =>This Inner Loop Header: Depth=1
    28		vmovss	(%rdi,%rcx,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    29		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    30		vmovss	%xmm2, (%rdi,%rcx,4)
    31		vmovss	4(%rdi,%rcx,4), %xmm2          

* details may be different, but you'll find something like this for clang (gcc generates a simpler code)
```
    27	.LBB0_8:                                # =>This Inner Loop Header: Depth=1
    28		vmovss	(%rdi,%rcx,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    29		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    30		vmovss	%xmm2, (%rdi,%rcx,4)
    31		vmovss	4(%rdi,%rcx,4), %xmm2           # xmm2 = mem[0],zero,zero,zero
    32		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    33		vmovss	%xmm2, 4(%rdi,%rcx,4)
    34		vmovss	8(%rdi,%rcx,4), %xmm2           # xmm2 = mem[0],zero,zero,zero
    35		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    36		vmovss	%xmm2, 8(%rdi,%rcx,4)
    37		vmovss	12(%rdi,%rcx,4), %xmm2          # xmm2 = mem[0],zero,zero,zero
    38		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    39		vmovss	%xmm2, 12(%rdi,%rcx,4)
    40		addq	$4, %rcx
    41		cmpq	%rcx, %rsi
    42		jne	.LBB0_8
```

* observe the label `.LBB0_8` and `jne .LBB0_8` (jump if not equal) form a loop
* `vfmadd132ss` does `a * x + b`, and importantly, `ss` is a hallmark of scalar instruction and single precision instruction
* the loop body contains four vfmadd213ss. it is the result of compiler optimization (loop unrolling) that reduces the number of compare / jump instructions to iterate
* with `vfmadd132ss`, we are pretty confident that the code is NOT vectorized

## 7-2. vectorized version
* now let's look at a vectorized version

In [16]:
%%writefile axpb_simd.c
void axpb_simd(float a, float * x, float b, long n) {
  asm volatile("# ========== loop begins ==========");
  for (long i = 0; i < n; i++) {
    x[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing axpb_simd.c


In [17]:
# will produce axpb_simd.s
clang -S -Wall -O3 -mavx512f -mfma -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpb_simd.c
#nvc -S -Wall -O3 -mavx512f -mfma -Minfo=vect -Mneginfo=vect axpb_simd.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopt-info-vec-optimized -fopt-info-vec-missed axpb_simd.c

axpb_simd.c:3:3: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
  for (long i = 0; i < n; i++) {
  ^


In [18]:
cat -n axpb_simd.s

     1		.text
     2		.file	"axpb_simd.c"
     3		.globl	axpb_simd                       # -- Begin function axpb_simd
     4		.p2align	4, 0x90
     5		.type	axpb_simd,@function
     6	axpb_simd:                              # @axpb_simd
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		testq	%rsi, %rsi
    13		jle	.LBB0_13
    14	# %bb.1:
    15		cmpq	$8, %rsi
    16		jae	.LBB0_3
    17	# %bb.2:
    18		xorl	%eax, %eax
    19		jmp	.LBB0_12
    20	.LBB0_3:
    21		cmpq	$64, %rsi
    22		jae	.LBB0_5
    23	# %bb.4:
    24		xorl	%eax, %eax
    25		jmp	.LBB0_9
    26	.LBB0_5:
    27		movq	%rsi, %rax
    28		andq	$-64, %rax
    29		vbroadcastss	%xmm0, %zmm2
    30		vbroadcastss	%xmm1, %zmm3
    31		xorl	%ecx, %ecx
    32		.p2align	4, 0x90
    33	.LBB0_6:                                # =>This Inner Loop Header: Depth=1
    34		vmovups	(%rdi,%rcx,4), %zmm4
    35		vfmadd213ps	%zmm3, %zmm2, %zmm4     # zmm4 = (zmm2 * zmm4

* you will find it difficult to locate where is the loop you are looking for
* look for labels (`.Lx`) and jump instructions (`jxx .Lx`) that forms a small loop, which will be something like
```
    33	.LBB0_6:                                # =>This Inner Loop Header: Depth=1
    34		vmovups	(%rdi,%rcx,4), %zmm4
    35		vfmadd213ps	%zmm3, %zmm2, %zmm4     # zmm4 = (zmm2 * zmm4) + zmm3
    36		vmovups	64(%rdi,%rcx,4), %zmm5
    37		vfmadd213ps	%zmm3, %zmm2, %zmm5     # zmm5 = (zmm2 * zmm5) + zmm3
    38		vmovups	128(%rdi,%rcx,4), %zmm6
    39		vfmadd213ps	%zmm3, %zmm2, %zmm6     # zmm6 = (zmm2 * zmm6) + zmm3
    40		vmovups	192(%rdi,%rcx,4), %zmm7
    41		vfmadd213ps	%zmm3, %zmm2, %zmm7     # zmm7 = (zmm2 * zmm7) + zmm3
    42		vmovups	%zmm4, (%rdi,%rcx,4)
    43		vmovups	%zmm5, 64(%rdi,%rcx,4)
    44		vmovups	%zmm6, 128(%rdi,%rcx,4)
    45		vmovups	%zmm7, 192(%rdi,%rcx,4)
    46		addq	$64, %rcx
    47		cmpq	%rcx, %rax
    48		jne	.LBB0_6
```
* it looks very similar to the previous example, but notice `vfmadd132ps` instruction, which is _p_acked (vectorized) _s_ingle-precision instruction.  this _p_ is the hallmark of vectorized code
* the generated instructions include many other instructions and look complex. this is because the loop count may not be a multiple of SIMD width, in which case there must be remainder scalar iterations before or after the vectorized iterations.  that is,
```
for (i = 0; i < n; i++) {
  S(i);
}
```
becomes something like
```
for (i = 0; i + L <= n; i += L) {
  S(i:i+L);
}
for (; i < n; i++) {
  S(i);
}
```

* just to make our investigations easier, let's eliminate the remainder iterations by forcibly making n a multiple of the largest possible SIMD width (16).  to be sure, this does not preserve the behavior of the original scalar loop.  we nevertheless do so just for the purpose of making the generated code simpler and easier to look into.
* `n = (n / 16) * 16` in the code below does the trick

In [19]:
%%writefile axpy_simd_no_remainder.c
void axpb_simd_no_remainder(float a, float * x, float b, long n) {
  n = (n / 16) * 16;    /* just so that there are no scalar iterations */
  asm volatile("# ========== loop begins ==========");
  for (long i = 0; i < n; i++) {
    x[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing axpy_simd_no_remainder.c


In [20]:
clang -S -Wall -O3 -mavx512f -mfma -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpy_simd_no_remainder.c
#nvc -S -Wall -O3 -mavx512f -mfma -Minfo=vect -Mneginfo=vect axpy_simd_no_remainder.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopt-info-vec-optimized -fopt-info-vec-missed axpy_simd_no_remainder.c

axpy_simd_no_remainder.c:4:3: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
  for (long i = 0; i < n; i++) {
  ^


In [21]:
cat -n axpy_simd_no_remainder.s

     1		.text
     2		.file	"axpy_simd_no_remainder.c"
     3		.globl	axpb_simd_no_remainder          # -- Begin function axpb_simd_no_remainder
     4		.p2align	4, 0x90
     5		.type	axpb_simd_no_remainder,@function
     6	axpb_simd_no_remainder:                 # @axpb_simd_no_remainder
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		cmpq	$16, %rsi
    13		jl	.LBB0_11
    14	# %bb.1:
    15		movq	%rsi, %rax
    16		andq	$-16, %rax
    17		je	.LBB0_2
    18	# %bb.4:
    19		cmpq	$64, %rax
    20		jae	.LBB0_6
    21	# %bb.5:
    22		xorl	%ecx, %ecx
    23		jmp	.LBB0_9
    24	.LBB0_2:
    25		xorl	%ecx, %ecx
    26		.p2align	4, 0x90
    27	.LBB0_3:                                # =>This Inner Loop Header: Depth=1
    28		vmovss	(%rdi,%rcx,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    29		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    30		vmovss	%xmm2, (%rdi,%rcx,4)
    31		incq	%rcx

# 8. When vectorization succeeds or fails
* as simple as it may sound, there are so many ways vectorization fails
* in this section, you will test many code examples we could hope compilers to vectorize and see if they actually do
* <font color="red">share what you witnessed and collaborate in a shared Excel workbook.</font>    Go [this page](https://univtokyo-my.sharepoint.com/:x:/g/personal/2615215597_utac_u-tokyo_ac_jp/EamIiDy6df5CodLocg_7Zu8BpyBJ6ki37Ngc_S9jZ2DYSA?e=kbmG5k) for detailed instructions.

## 8-1. Dependencies
* the most typical is a loop that has _dependencies_ between iterations, so executing them simultaneously changes the behavior of the loop
* a compiler fails to vectorize this code

In [22]:
%%writefile dependency.c
void dependency(float a, float * x, float b, long n) {
  asm volatile("# ========== loop begins ==========");
  for (long i = 0; i < n - 1; i++) {
    x[i+1] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing dependency.c


* see the compiler message to see if the code is vectorized
* note: GCC generates an informative message that explain why it failed to vectorize it

In [23]:
clang -S -Wall -O3 -mavx512f -mfma -Rpass=loop-vectorize -Rpass-missed=loop-vectorize dependency.c
#nvc -S -Wall -O3 -mavx512f -mfma -Minfo=vect -Mneginfo=vect dependency.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopt-info-vec-optimized -fopt-info-vec-missed dependency.c

dependency.c:3:3: remark: loop not vectorized [-Rpass-missed=loop-vectorize]
  for (long i = 0; i < n - 1; i++) {
  ^


* the reason is that the vectorization of the loop 
```
for (long i = 0; i < n - 1; i += L) {
  x[i+1:i+1+L] = a * x[i:i+L] + b;
}
```
would change the behavior of the original loop.
* to illustrate, let's say the SIMD width L is 4. the first iteration of the vectorized loop does
```
x[1:5] = a * x[0:4] + b;
```
whereas the corresponding first four iterations of the original code performs
```
i=0 : x[1] = a * x[0] + b;
i=1 : x[2] = a * x[1] + b;
i=2 : x[3] = a * x[2] + b;
i=3 : x[4] = a * x[3] + b;
```
* observe that the second iteration (i=1) reads x[1], which is _updated_ in the first iteration (i=0). it does not happen in the vectorized version

* therefore _the two versions are not equivalent_
* generally speaking, conditions in which a loop can be vectorized is similar to the conditions in which a loop can be parallelized. after all, both execute multiple iterations simultaneously
* looking at the original code again,
```
for (long i = 0; i < n - 1; i++) {
  x[i+1] = a * x[i] + b;
}
```
the compiler did not vectorize it because different iterations (say i=0 and i=1) read the same element (x[1]) and at least one of them is a write (i=0 writes to x[1])

## 8-2. Potential (uncertain) dependencies
* a similar but less obvious example is this
* it looks like adding two arrays x and y and putting the result into a third array z, so is safe to vectorize, but what if z is overlapping with x (say &z[0] = &x[1])
* then it would be equivalent to
```
x[i+1] = x[i] + y[i]
```
creating exactly the same problem as the previous one.
* try and observe how the compiler responds

In [24]:
%%writefile uncertain_dependency.c
void uncertain_dependency(float a, float * x, float b, float * y, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing uncertain_dependency.c


In [25]:
clang -S -Wall -O3 -mavx512f -mfma -Rpass=loop-vectorize -Rpass-missed=loop-vectorize uncertain_dependency.c
#nvc -S -Wall -O3 -mavx512f -mfma -Minfo=vect -Mneginfo=vect uncertain_dependency.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopt-info-vec-optimized -fopt-info-vec-missed uncertain_dependency.c

uncertain_dependency.c:4:3: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
  for (long i = 0; i < n; i++) {
  ^


In [26]:
cat -n uncertain_dependency.s

     1		.text
     2		.file	"uncertain_dependency.c"
     3		.globl	uncertain_dependency            # -- Begin function uncertain_dependency
     4		.p2align	4, 0x90
     5		.type	uncertain_dependency,@function
     6	uncertain_dependency:                   # @uncertain_dependency
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		cmpq	$16, %rdx
    13		jl	.LBB0_12
    14	# %bb.1:
    15		movq	%rdx, %r8
    16		andq	$-16, %r8
    17		je	.LBB0_3
    18	# %bb.2:
    19		movq	%rsi, %rax
    20		subq	%rdi, %rax
    21		cmpq	$255, %rax
    22		jbe	.LBB0_3
    23	# %bb.5:
    24		cmpq	$64, %r8
    25		jae	.LBB0_7
    26	# %bb.6:
    27		xorl	%ecx, %ecx
    28		jmp	.LBB0_10
    29	.LBB0_3:
    30		xorl	%ecx, %ecx
    31		.p2align	4, 0x90
    32	.LBB0_4:                                # =>This Inner Loop Header: Depth=1
    33		vmovss	(%rdi,%rcx,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    34		vfmadd213ss	%xmm1, %x

## 8-3. `#pragma omp simd` directive
* `#pragma omp simd` directive tells the compiler to vectorize it without worrying about these dependency issues
* https://www.openmp.org/spec-html/5.0/openmpsu42.html#x65-1390002.9.3
* you need to pass compiler options to let the compiler recognize the directive
 * clang(++) : `-fopenmp` (enable OpenMP) or `-fopenmp-simd`
 * nvc(++) : `-mp` (enable OpenMP)
 * clang(++) : `-fopenmp` (enable OpenMP) or `-fopenmp-simd`


In [27]:
%%writefile axpy_omp_simd.c
void axpb_omp_simd(float a, float * x, float b, float * y, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing axpy_omp_simd.c


In [28]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize axpy_omp_simd.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect axpy_omp_simd.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed axpy_omp_simd.c

axpy_omp_simd.c:4:1: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
#pragma omp simd
^


In [29]:
cat -n axpy_omp_simd.s

     1		.text
     2		.file	"axpy_omp_simd.c"
     3		.globl	axpb_omp_simd                   # -- Begin function axpb_omp_simd
     4		.p2align	4, 0x90
     5		.type	axpb_omp_simd,@function
     6	axpb_omp_simd:                          # @axpb_omp_simd
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		cmpq	$16, %rdx
    13		jl	.LBB0_11
    14	# %bb.1:
    15		movq	%rdx, %r8
    16		andq	$-16, %r8
    17		je	.LBB0_2
    18	# %bb.4:
    19		cmpq	$64, %r8
    20		jae	.LBB0_6
    21	# %bb.5:
    22		xorl	%ecx, %ecx
    23		jmp	.LBB0_9
    24	.LBB0_2:
    25		xorl	%eax, %eax
    26		.p2align	4, 0x90
    27	.LBB0_3:                                # =>This Inner Loop Header: Depth=1
    28		vmovss	(%rdi,%rax,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    29		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    30		vmovss	%xmm2, (%rsi,%rax,4)
    31		incq	%rax
    32		cmpq	%rax, %r8
    33		jne	.LB

* in the interest in maximizing the possibility of vectorization, we have `#pragma omp simd` in all the examples below
* it's not that then the compiler is able to vectorize any kind of code, as you will see shortly

## 8-4. reduction
* just like parallelization, reduction is a common pattern that superficially introduces dependencies but can actually be vectorized by a suitable execution strategy
* for example,
```
s = 0.0;
for (i = 0; i < n; i++) {
  s += x[i];
}
```
can be executed like
```
sv = {0.0, 0.0, ..., 0.0};
for (i = 0; i < n; i++) {
  sv += x[i:i+L];
}
s = sv[0] + ... + sv[L-1];
```

In [30]:
%%writefile sum.c
float sum(float * x, long n) {
  n = (n / 16) * 16;
  float s = 0.0;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd reduction(+:s)
  for (long i = 0; i < n; i++) {
    s += x[i];
  }
  asm volatile("# ---------- loop ends ----------");
  return s;
}

Writing sum.c


In [31]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize sum.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect sum.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed sum.c

sum.c:5:1: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
#pragma omp simd reduction(+:s)
^


In [32]:
cat -n sum.s

     1		.text
     2		.file	"sum.c"
     3		.section	.rodata.cst4,"aM",@progbits,4
     4		.p2align	2                               # -- Begin function sum
     5	.LCPI0_0:
     6		.long	0x80000000                      # float -0
     7		.section	.rodata,"a",@progbits
     8		.p2align	6
     9	.LCPI0_1:
    10		.long	0x00000000                      # float 0
    11		.long	0x80000000                      # float -0
    12		.long	0x80000000                      # float -0
    13		.long	0x80000000                      # float -0
    14		.long	0x80000000                      # float -0
    15		.long	0x80000000                      # float -0
    16		.long	0x80000000                      # float -0
    17		.long	0x80000000                      # float -0
    18		.long	0x80000000                      # float -0
    19		.long	0x80000000                      # float -0
    20		.long	0x80000000                      # float -0
    21		.long	0x80000000                      # float -0
    22		.lon

## 8-5. if expression within a loop
* we are going to see a number of typical construct that hampers vectorization
* bear in mind that there are no universal rules, however
* whether a particular code is vectorized or not depends on the compiler and the available instructions

* does an if statement within a loop hamper vectorization?

In [33]:
%%writefile branch.c
void branch(float a, float * x, float b, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    if (i % 2 == 0) {
      x[i] = a * x[i] + b;
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing branch.c


In [34]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize branch.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect branch.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed branch.c

branch.c:4:1: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
#pragma omp simd
^


In [35]:
cat -n branch.s

     1		.text
     2		.file	"branch.c"
     3		.section	.rodata,"a",@progbits
     4		.p2align	6                               # -- Begin function branch
     5	.LCPI0_0:
     6		.quad	8                               # 0x8
     7		.quad	9                               # 0x9
     8		.quad	10                              # 0xa
     9		.quad	11                              # 0xb
    10		.quad	12                              # 0xc
    11		.quad	13                              # 0xd
    12		.quad	14                              # 0xe
    13		.quad	15                              # 0xf
    14	.LCPI0_1:
    15		.quad	0                               # 0x0
    16		.quad	1                               # 0x1
    17		.quad	2                               # 0x2
    18		.quad	3                               # 0x3
    19		.quad	4                               # 0x4
    20		.quad	5                               # 0x5
    21		.quad	6                               # 0x6
    22		.quad	7                 

* recent Intel CPUs have predicated (masked) SIMD instructions so it should actually be possible to vectorize this code

## 8-6. a loop in a loop (a constant trip count)
* an inner loop inside an outer loop which we want to vectorize
* the trip count of the inner loop is a compile-time constant 

In [36]:
%%writefile loop_c.c
/* inner loop, with a compile-time constant trip count */
void loop_c(float a, float * x, float b, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    for (long k = 0; k < 10; k++) {
      x[i] = a * x[i] + b;
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing loop_c.c


In [37]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize loop_c.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect loop_c.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed loop_c.c

loop_c.c:5:1: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
#pragma omp simd
^


In [38]:
cat -n loop_c.s

     1		.text
     2		.file	"loop_c.c"
     3		.globl	loop_c                          # -- Begin function loop_c
     4		.p2align	4, 0x90
     5		.type	loop_c,@function
     6	loop_c:                                 # @loop_c
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		cmpq	$16, %rsi
    13		jl	.LBB0_11
    14	# %bb.1:
    15		movq	%rsi, %rax
    16		andq	$-16, %rax
    17		je	.LBB0_2
    18	# %bb.4:
    19		cmpq	$64, %rax
    20		jae	.LBB0_6
    21	# %bb.5:
    22		xorl	%ecx, %ecx
    23		jmp	.LBB0_9
    24	.LBB0_2:
    25		xorl	%ecx, %ecx
    26		.p2align	4, 0x90
    27	.LBB0_3:                                # =>This Inner Loop Header: Depth=1
    28		vmovss	(%rdi,%rcx,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    29		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    30		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    31		vfmadd213ss	%xmm1, %xmm0, %xmm2    

## 8-7. a loop in a loop (a variable but loop-invariant trip count)
* the same as above, but with the trip count of the inner loop is a variable
* it is constant across all iterations of the outer loop, so the entire iteration space is a rectangle and it should be easily recognized as such

In [39]:
%%writefile loop_i.c
/* inner loop, with a loop-invariant trip count */
void loop_i(float a, float * x, float b, long n, long m) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    for (int k = 0; k < m; k++) {
      x[i] = a * x[i] + b;
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing loop_i.c


In [40]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize loop_i.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect loop_i.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed loop_i.c

loop_i.c:7:5: remark: loop not vectorized [-Rpass-missed=loop-vectorize]
    for (int k = 0; k < m; k++) {
    ^
loop_i.c:5:1: warning: loop not vectorized: the optimizer was unable to perform the requested transformation; the transformation might be disabled or specified as part of an unsupported transformation ordering [-Wpass-failed=transform-warning]
#pragma omp simd
^
1 warning generated.


In [41]:
cat -n loop_i.s

     1		.text
     2		.file	"loop_i.c"
     3		.globl	loop_i                          # -- Begin function loop_i
     4		.p2align	4, 0x90
     5		.type	loop_i,@function
     6	loop_i:                                 # @loop_i
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		cmpq	$16, %rsi
    13		jl	.LBB0_10
    14	# %bb.1:
    15		testq	%rdx, %rdx
    16		jle	.LBB0_10
    17	# %bb.2:
    18		leaq	15(%rsi), %rax
    19		testq	%rsi, %rsi
    20		cmovnsq	%rsi, %rax
    21		andq	$-16, %rax
    22		cmpq	$2, %rax
    23		movl	$1, %r9d
    24		cmovgeq	%rax, %r9
    25		leaq	-1(%rdx), %r8
    26		movl	%edx, %esi
    27		andl	$7, %esi
    28		andq	$-8, %rdx
    29		xorl	%ecx, %ecx
    30		jmp	.LBB0_3
    31		.p2align	4, 0x90
    32	.LBB0_9:                                #   in Loop: Header=BB0_3 Depth=1
    33		vmovss	%xmm2, (%rdi,%rcx,4)
    34		incq	%rcx
    35		cmpq	%r9, %rcx
    36		je	.LBB0_10
    37	.LBB0_3:         

## 8-8. a loop within a loop (variable trip counts)
* the same as above, but with the trip count of the inner loop being different from one iteration to another
* therefore the entire iteration space is not a rectangle

In [42]:
%%writefile loop_v.c
/* inner loop, with a variable trip count */
void loop_v(float a, float * x, float b, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    for (int k = 0; k < i; k++) {
      x[i] = a * x[i] + b;
    }
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing loop_v.c


In [43]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize loop_v.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect loop_v.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed loop_v.c

loop_v.c:7:5: remark: loop not vectorized [-Rpass-missed=loop-vectorize]
    for (int k = 0; k < i; k++) {
    ^
loop_v.c:5:1: warning: loop not vectorized: the optimizer was unable to perform the requested transformation; the transformation might be disabled or specified as part of an unsupported transformation ordering [-Wpass-failed=transform-warning]
#pragma omp simd
^
1 warning generated.


In [44]:
cat -n loop_v.s

     1		.text
     2		.file	"loop_v.c"
     3		.globl	loop_v                          # -- Begin function loop_v
     4		.p2align	4, 0x90
     5		.type	loop_v,@function
     6	loop_v:                                 # @loop_v
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		cmpq	$16, %rsi
    13		jl	.LBB0_11
    14	# %bb.1:
    15		andq	$-16, %rsi
    16		xorl	%eax, %eax
    17		jmp	.LBB0_2
    18		.p2align	4, 0x90
    19	.LBB0_9:                                #   in Loop: Header=BB0_2 Depth=1
    20		vmovss	%xmm2, (%rdi,%rax,4)
    21	.LBB0_10:                               #   in Loop: Header=BB0_2 Depth=1
    22		incq	%rax
    23		cmpq	%rsi, %rax
    24		je	.LBB0_11
    25	.LBB0_2:                                # =>This Loop Header: Depth=1
    26	                                        #     Child Loop BB0_5 Depth 2
    27	                                        #     Child Loop BB0_8 Depth 2
    28		testq	%ra

## 8-9. a function call within a loop
* a function call within a loop
* without the body of the function visible to the compiler compiling the loop (i.e., not in the same compilation unit as the loop), we cannot have a vectorized version of the function
* therefore it is highly unlikely that this code gets meaningfully vectorized

In [45]:
%%writefile funcall.c
float f(float a, float x, float b);
void funcall(float a, float * x, float b, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    x[i] = f(a, x[i], b);
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing funcall.c


In [46]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize funcall.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect funcall.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed funcall.c

funcall.c:5:1: remark: loop not vectorized (Force=true) [-Rpass-missed=loop-vectorize]
#pragma omp simd
^
funcall.c:5:1: warning: loop not vectorized: the optimizer was unable to perform the requested transformation; the transformation might be disabled or specified as part of an unsupported transformation ordering [-Wpass-failed=transform-warning]
1 warning generated.


In [47]:
cat -n funcall.s

     1		.text
     2		.file	"funcall.c"
     3		.globl	funcall                         # -- Begin function funcall
     4		.p2align	4, 0x90
     5		.type	funcall,@function
     6	funcall:                                # @funcall
     7		.cfi_startproc
     8	# %bb.0:
     9		pushq	%r15
    10		.cfi_def_cfa_offset 16
    11		pushq	%r14
    12		.cfi_def_cfa_offset 24
    13		pushq	%rbx
    14		.cfi_def_cfa_offset 32
    15		subq	$16, %rsp
    16		.cfi_def_cfa_offset 48
    17		.cfi_offset %rbx, -32
    18		.cfi_offset %r14, -24
    19		.cfi_offset %r15, -16
    20		#APP
    21		# ========== loop begins ==========
    22		#NO_APP
    23		cmpq	$16, %rsi
    24		jl	.LBB0_3
    25	# %bb.1:
    26		movq	%rsi, %r14
    27		movq	%rdi, %r15
    28		andq	$-16, %r14
    29		xorl	%ebx, %ebx
    30		vmovss	%xmm1, 12(%rsp)                 # 4-byte Spill
    31		vmovss	%xmm0, 8(%rsp)                  # 4-byte Spill
    32		.p2align	4, 0x90
    33	.LBB0_2:                                # =>This Inner

## 8-10. a function call within a loop with `#pragma omp declare simd`
* OpenMP has a directive `#pragma omp declare simd`, which says there is a vectorized version of a function
* https://www.openmp.org/spec-html/5.0/openmpsu42.html#x65-1390002.9.3
* calling such a function within a loop does not hamper vectorization, even if the body of the function is not in the same compilation unit as the loop

In [48]:
%%writefile funcall_decl_simd.c
#pragma omp declare simd uniform(a, b) notinbranch
float f(float a, float x, float b);

void funcall_decl_simd(float a, float * x, float b, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    x[i] = f(a, x[i], b);
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing funcall_decl_simd.c


In [49]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize funcall_decl_simd.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect funcall_decl_simd.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed funcall_decl_simd.c

funcall_decl_simd.c:7:1: remark: loop not vectorized (Force=true) [-Rpass-missed=loop-vectorize]
#pragma omp simd
^
funcall_decl_simd.c:7:1: warning: loop not vectorized: the optimizer was unable to perform the requested transformation; the transformation might be disabled or specified as part of an unsupported transformation ordering [-Wpass-failed=transform-warning]
1 warning generated.


In [50]:
cat -n funcall_decl_simd.s

     1		.text
     2		.file	"funcall_decl_simd.c"
     3		.globl	funcall_decl_simd               # -- Begin function funcall_decl_simd
     4		.p2align	4, 0x90
     5		.type	funcall_decl_simd,@function
     6	funcall_decl_simd:                      # @funcall_decl_simd
     7		.cfi_startproc
     8	# %bb.0:
     9		pushq	%r15
    10		.cfi_def_cfa_offset 16
    11		pushq	%r14
    12		.cfi_def_cfa_offset 24
    13		pushq	%rbx
    14		.cfi_def_cfa_offset 32
    15		subq	$16, %rsp
    16		.cfi_def_cfa_offset 48
    17		.cfi_offset %rbx, -32
    18		.cfi_offset %r14, -24
    19		.cfi_offset %r15, -16
    20		#APP
    21		# ========== loop begins ==========
    22		#NO_APP
    23		cmpq	$16, %rsi
    24		jl	.LBB0_3
    25	# %bb.1:
    26		movq	%rsi, %r14
    27		movq	%rdi, %r15
    28		andq	$-16, %r14
    29		xorl	%ebx, %ebx
    30		vmovss	%xmm1, 12(%rsp)                 # 4-byte Spill
    31		vmovss	%xmm0, 8(%rsp)                  # 4-byte Spill
    32		.p2align	4, 0x90
    33	.LBB0_2:      

## 8-11. a function defined with `#pragma omp declare simd`
* what happens on the function defined with `#pragma omp declare simd` ?

In [51]:
%%writefile fundef_decl_simd.c
#pragma omp declare simd uniform(a, b) notinbranch
float fundef_decl_simd(float a, float x, float b) {
  return a * x + b;
}

Writing fundef_decl_simd.c


In [52]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize fundef_decl_simd.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect fundef_decl_simd.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed fundef_decl_simd.c

In [53]:
cat -n fundef_decl_simd.s

     1		.text
     2		.file	"fundef_decl_simd.c"
     3		.globl	fundef_decl_simd                # -- Begin function fundef_decl_simd
     4		.p2align	4, 0x90
     5		.type	fundef_decl_simd,@function
     6	fundef_decl_simd:                       # @fundef_decl_simd
     7		.cfi_startproc
     8	# %bb.0:
     9		vfmadd213ss	%xmm2, %xmm1, %xmm0     # xmm0 = (xmm1 * xmm0) + xmm2
    10		retq
    11	.Lfunc_end0:
    12		.size	fundef_decl_simd, .Lfunc_end0-fundef_decl_simd
    13		.cfi_endproc
    14	                                        # -- End function
    15		.ident	"clang version 15.0.0"
    16		.section	".note.GNU-stack","",@progbits
    17		.addrsig


## 8-12. a strided array access (load)
* an ordinary vectorized load/store instruction (vmovups or vmovaps) takes a pointer (address) and reads or writes a consecutive set of addresses
* for example, the following load instruction
```
vmovups (%rax),%zmm0
```
reads the consecutive 64 bytes starting from the address in the `rax` register and gets the result into `zmm0` register
* only with this instruction it is difficult to vectorize a loop whose consecutive iterations does not read the consecutive address
* the simplest of which is a strided access like this

* two possible strategies
  * access arrays using ordinary vector load instructions and shuffle data around on registers
  * access arrays using `gather` instructions

In [54]:
%%writefile stride_load.c
void stride_load(float a, float * x, float b, float * y, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i * 2] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing stride_load.c


In [55]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize stride_load.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect stride_load.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed stride_load.c

stride_load.c:4:1: remark: vectorized loop (vectorization width: 16, interleaved count: 4) [-Rpass=loop-vectorize]
#pragma omp simd
^


In [56]:
cat -n stride_load.s

     1		.text
     2		.file	"stride_load.c"
     3		.section	.rodata,"a",@progbits
     4		.p2align	6                               # -- Begin function stride_load
     5	.LCPI0_0:
     6		.long	0                               # 0x0
     7		.long	2                               # 0x2
     8		.long	4                               # 0x4
     9		.long	6                               # 0x6
    10		.long	8                               # 0x8
    11		.long	10                              # 0xa
    12		.long	12                              # 0xc
    13		.long	14                              # 0xe
    14		.long	16                              # 0x10
    15		.long	18                              # 0x12
    16		.long	20                              # 0x14
    17		.long	22                              # 0x16
    18		.long	24                              # 0x18
    19		.long	26                              # 0x1a
    20		.long	28                              # 0x1c
    21		.long	30                

## 8-13. a strided array access (store)

In [57]:
%%writefile stride_store.c
void stride_store(float a, float * x, float b, float * y, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    y[i * 2] = a * x[i] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing stride_store.c


In [58]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize stride_store.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect stride_store.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed stride_store.c

stride_store.c:4:1: remark: vectorized loop (vectorization width: 16, interleaved count: 1) [-Rpass=loop-vectorize]
#pragma omp simd
^


In [59]:
cat -n stride_store.s

     1		.text
     2		.file	"stride_store.c"
     3		.section	.rodata,"a",@progbits
     4		.p2align	6                               # -- Begin function stride_store
     5	.LCPI0_0:
     6		.quad	8                               # 0x8
     7		.quad	9                               # 0x9
     8		.quad	10                              # 0xa
     9		.quad	11                              # 0xb
    10		.quad	12                              # 0xc
    11		.quad	13                              # 0xd
    12		.quad	14                              # 0xe
    13		.quad	15                              # 0xf
    14	.LCPI0_1:
    15		.quad	0                               # 0x0
    16		.quad	1                               # 0x1
    17		.quad	2                               # 0x2
    18		.quad	3                               # 0x3
    19		.quad	4                               # 0x4
    20		.quad	5                               # 0x5
    21		.quad	6                               # 0x6
    22		.quad	7     

## 8-14. an array of structures
* a less obvious and commonly occurring pattern is to access an array of structures
* even if the array indices of consecutive iterations are consecutive, actual addresses are not

In [60]:
%%writefile struct_load.c
typedef struct {
  float x;
  float y;
} point_t;
void struct_load(float a, point_t * p, float b, point_t * q, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
#pragma omp simd
  for (long i = 0; i < n; i++) {
    q[i].x = a * p[i].x + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing struct_load.c


In [62]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize struct_load.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect struct_load.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed struct_load.c

struct_load.c:8:1: remark: vectorized loop (vectorization width: 16, interleaved count: 1) [-Rpass=loop-vectorize]
#pragma omp simd
^


In [63]:
cat -n struct_load.s

     1		.text
     2		.file	"struct_load.c"
     3		.section	.rodata,"a",@progbits
     4		.p2align	6                               # -- Begin function struct_load
     5	.LCPI0_0:
     6		.quad	8                               # 0x8
     7		.quad	9                               # 0x9
     8		.quad	10                              # 0xa
     9		.quad	11                              # 0xb
    10		.quad	12                              # 0xc
    11		.quad	13                              # 0xd
    12		.quad	14                              # 0xe
    13		.quad	15                              # 0xf
    14	.LCPI0_1:
    15		.quad	0                               # 0x0
    16		.quad	1                               # 0x1
    17		.quad	2                               # 0x2
    18		.quad	3                               # 0x3
    19		.quad	4                               # 0x4
    20		.quad	5                               # 0x5
    21		.quad	6                               # 0x6
    22		.quad	7       

## 8-15. an irregular array access
* irregular array accesses, in which consecutive iterations access elements with non-constant strides

In [64]:
%%writefile non_affine_idx.c
void non_affine_idx(float a, float * x, float b, float * y, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i * i % n] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing non_affine_idx.c


In [65]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize non_affine_idx.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect non_affine_idx.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed non_affine_idx.c

non_affine_idx.c:4:3: remark: loop not vectorized [-Rpass-missed=loop-vectorize]
  for (long i = 0; i < n; i++) {
  ^


In [66]:
cat -n non_affine_idx.s

     1		.text
     2		.file	"non_affine_idx.c"
     3		.globl	non_affine_idx                  # -- Begin function non_affine_idx
     4		.p2align	4, 0x90
     5		.type	non_affine_idx,@function
     6	non_affine_idx:                         # @non_affine_idx
     7		.cfi_startproc
     8	# %bb.0:
     9		pushq	%rbx
    10		.cfi_def_cfa_offset 16
    11		.cfi_offset %rbx, -16
    12		leaq	15(%rdx), %r8
    13		testq	%rdx, %rdx
    14		cmovnsq	%rdx, %r8
    15		#APP
    16		# ========== loop begins ==========
    17		#NO_APP
    18		cmpq	$16, %rdx
    19		jl	.LBB0_9
    20	# %bb.1:
    21		andq	$-16, %r8
    22		cmpq	$2, %r8
    23		movl	$1, %r10d
    24		cmovgeq	%r8, %r10
    25		movabsq	$9223372036854775792, %r11      # imm = 0x7FFFFFFFFFFFFFF0
    26		andq	%r10, %r11
    27		movl	$4, %ebx
    28		xorl	%ecx, %ecx
    29		xorl	%eax, %eax
    30		jmp	.LBB0_2
    31		.p2align	4, 0x90
    32	.LBB0_13:                               #   in Loop: Header=BB0_2 Depth=1
    33		xorl	%edx, %edx
  

## 8-16. an indirect array access
* the same as above, but with indices determined by another index array, a frequently occurring pattern in sparse matrix and graph applications

In [67]:
%%writefile indirect_idx.c
void indirect_idx(float a, float * x, long * idx, float b, float * y, long n) {
  n = (n / 16) * 16;
  asm volatile("# ========== loop begins ==========");
  for (long i = 0; i < n; i++) {
    y[i] = a * x[idx[i]] + b;
  }
  asm volatile("# ---------- loop ends ----------");
}

Writing indirect_idx.c


In [68]:
clang -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -Rpass=loop-vectorize -Rpass-missed=loop-vectorize indirect_idx.c
#nvc -S -Wall -O3 -mavx512f -mfma -mp -Minfo=vect -Mneginfo=vect indirect_idx.c
#gcc -S -Wall -O3 -mavx512f -mfma -fopenmp-simd -fopt-info-vec-optimized -fopt-info-vec-missed indirect_idx.c

indirect_idx.c:4:3: remark: loop not vectorized [-Rpass-missed=loop-vectorize]
  for (long i = 0; i < n; i++) {
  ^


In [69]:
cat -n indirect_idx.s

     1		.text
     2		.file	"indirect_idx.c"
     3		.globl	indirect_idx                    # -- Begin function indirect_idx
     4		.p2align	4, 0x90
     5		.type	indirect_idx,@function
     6	indirect_idx:                           # @indirect_idx
     7		.cfi_startproc
     8	# %bb.0:
     9		#APP
    10		# ========== loop begins ==========
    11		#NO_APP
    12		cmpq	$16, %rcx
    13		jl	.LBB0_3
    14	# %bb.1:
    15		andq	$-16, %rcx
    16		xorl	%r8d, %r8d
    17		.p2align	4, 0x90
    18	.LBB0_2:                                # =>This Inner Loop Header: Depth=1
    19		movq	(%rsi,%r8,8), %rax
    20		vmovss	(%rdi,%rax,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    21		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    22		vmovss	%xmm2, (%rdx,%r8,4)
    23		movq	8(%rsi,%r8,8), %rax
    24		vmovss	(%rdi,%rax,4), %xmm2            # xmm2 = mem[0],zero,zero,zero
    25		vfmadd213ss	%xmm1, %xmm0, %xmm2     # xmm2 = (xmm0 * xmm2) + xmm1
    26		vmovss	%xmm2